In [1]:
!pip install pandas 

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 1.5 MB/s eta 0:00:06
   ---- ----------------------------------- 1.0/9.7 MB 1.7 MB/s eta 0:00:06
   ------ --------------------------------- 1.6/9.7 MB 1.8 MB/s eta 0:00:05
   ------- -------------------------------- 1.8/9.7 MB 1.8 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/9.7 MB 1.8 MB/s eta 0:00:05
   --------- ------------------------------ 2.4/9.7 MB 1.6 MB/s eta 0:00:05
   ---------- ----------------------------- 2.6/9.7 MB 1.5 MB/s eta 0:00:05
   ----------- ---------------------------- 2.9/9.7 MB 1.4 MB/s eta 0:00:05
   ------------ --------------------------- 3.1


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
print("Hello")

Hello


In [3]:
import pandas as pd

csv_path = "SurvayResult-GeneralData.csv"

df = pd.read_csv(csv_path, encoding="utf-8-sig")

student_col = "Student ID (学籍番号)"
df["Student ID_表示用"] = df[student_col].fillna("").astype(str).str.strip()
df.loc[df["Student ID_表示用"] == "", "Student ID_表示用"] = "(空欄)"

summary = (
    df.groupby("Student ID_表示用", dropna=False)
      .agg(
          出現回数=("ID", "size"),
          対応ID=("ID", lambda x: ", ".join(x.astype(str)))
      )
      .reset_index()
      .rename(columns={"Student ID_表示用": "Student ID"})
)

summary = summary.sort_values(
    by=["出現回数", "Student ID"],
    ascending=[False, True]
).reset_index(drop=True)

summary


,Student ID,出現回数,対応ID
0,4,2,"P18, P11"
1,62205915,2,"P7, P16"
2,62206242,2,"P9, P6"
3,62209443,2,"P11, P4"
4,62213742,2,"P4, P5"
5,62214418,2,"P5, P9"
6,62216404,2,"P13, P7"
7,82346968,2,"P3, P3"
8,82418785,2,"P1, P12"
9,82420414,2,"P23, P15"


In [ ]:
import pandas as pd

input_path = "Result_Old_orig.csv"
output_path = "Result_Old.csv"

# 元データを読み込む
df = pd.read_csv(input_path, encoding="utf-8-sig")

# この行より前のデータを削除する
# 42 2025/06/20 19:39:03  P1  1  2
target_timestamp = "2025/06/20 19:39:03"
target_id = "P1"
target_condition = "1"
target_first_answer = "2"

# 先頭4列: タイムスタンプ, ID, Condition, 1問目の回答
target_mask = (
    (df.iloc[:, 0].astype(str).str.strip() == target_timestamp) &
    (df.iloc[:, 1].astype(str).str.strip() == target_id) &
    (df.iloc[:, 2].astype(str).str.strip() == target_condition) &
    (df.iloc[:, 3].astype(str).str.strip() == target_first_answer)
)

if not target_mask.any():
    raise ValueError("指定した基準行が見つかりませんでした。")

start_index = target_mask[target_mask].index[0]

# 基準行を含めて、それ以降のデータだけ残す
result = df.loc[start_index:].reset_index(drop=True)

# CSVとして保存
result.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"元データ: {len(df)} 行")
print(f"削除後: {len(result)} 行")
print(f"保存先: {output_path}")

result.head()
